# 01b — Synthesize EN→puhekieli pairs (back-translation)

Rap lyrics are the register we want but are **FI-only**. OpenSubtitles gives real
EN→FI pairs, but its Finnish is only *mildly* colloquial — so a model trained on it
drifts toward safe, semi-formal Finnish. To teach it to **produce hard rap-register
puhekieli from English**, we manufacture the missing English side:

```
real FI lyric  --(local LLM: FI→EN)-->  synthetic EN
training pair  =  (synthetic EN  →  real FI lyric)
```

The **Finnish side is always the authentic lyric**, so the target the model learns
to generate is genuine puhekieli. Only the English input is synthetic (input noise
matters far less than output noise). This is standard low-resource MT practice.

Runs against **LM Studio** (OpenAI-compatible, `http://localhost:1234/v1`). Nothing
leaves the machine. Load a model in LM Studio and start its server first.

In [ ]:
from puhekieli_llm.config import LMSTUDIO_BASE_URL, LMSTUDIO_MODEL, CLEAN
from puhekieli_llm import synth
print('endpoint:', LMSTUDIO_BASE_URL)
print('model   :', LMSTUDIO_MODEL, '(override via LMSTUDIO_MODEL env)')
# quick connectivity + quality probe on one line
try:
    demo = synth.back_translate_line('mä oon stadin kundi ja me mennään keikalle')
    print('FI->EN probe:', demo)
except Exception as e:
    print('LM Studio not reachable:', e)
    print('Start LM Studio, load a model, enable the local server, then re-run.')

## Generate pairs

Resumable: skips ids already in `rap_synthetic.jsonl` and flushes every 25 lines,
so a crash/interrupt won't lose progress. Start with a small `limit` to eyeball
quality before running the whole set.

In [ ]:
# start small; raise once the output looks good
if (CLEAN / 'genius_rap.jsonl').exists():
    try:
        synth.synthesize_pairs(limit=100)
    except Exception as e:
        print('back-translation skipped (LM Studio not reachable?):', e)
else:
    print('No genius_rap.jsonl yet — run 01_collect.ipynb (fetch + clean) first.')

In [ ]:
# eyeball the synthetic pairs + confirm FI side stays puhekieli
from puhekieli_llm.sources import read_jsonl
from puhekieli_llm.eval import puhekieli_score, corpus_puhekieli_rate
p = CLEAN / 'rap_synthetic.jsonl'
if p.exists():
    recs = list(read_jsonl(p))
    print(f'{len(recs)} synthetic pairs\n')
    for r in recs[:8]:
        print(f"  EN (synth): {r['en']}")
        print(f"  FI (real) : [{puhekieli_score(r['fi'])['score']:+.1f}] {r['fi']}\n")
    print(f"FI side spoken-leaning fraction: {corpus_puhekieli_rate([r['fi'] for r in recs]):.0%}")
else:
    print('No synthetic pairs yet — run the cell above.')

## ✅ Next
Once the pairs look reasonable, raise `limit` (e.g. to all lyrics), let it run,
then head to `02_tokenizer.ipynb`. Training data will be:
`opensubtitles_enfi` (base ability) + `rap_synthetic` (puhekieli push).